In [ ]:
import pandas as pd
from tqdm.auto import tqdm
from parquet_files import REDDIT_WSB_CACHE
tqdm.pandas()

df = pd.read_parquet(REDDIT_WSB_CACHE)
df.head()


,eastern,text,mentioned_tickers
1,2021-01-28 06:32:10-05:00,Math Professor Scott Steiner says the numbers ...,[GME]
2,2021-01-28 06:30:35-05:00,Exit the system The CEO of NASDAQ pushed to ha...,"[GME, QQQ]"
3,2021-01-28 06:28:57-05:00,NEW SEC FILING FOR GME! CAN SOMEONE LESS RETAR...,[GME]
4,2021-01-28 06:26:56-05:00,"Not to distract from GME, just thought our AMC...","[GME, AMC]"
6,2021-01-28 06:26:27-05:00,SHORT STOCK DOESN'T HAVE AN EXPIRATION DATE He...,[T]


In [16]:
bullish = {
    "moon", "mooning", "tendies", "tendy", "rocket", "rockets",
    "rip", "ripping", "face-ripper", "face ripper",
    "squeeze", "short squeeze", "mega squeeze", "gamma squeeze",
    "yolo", "yoloing",
    "up only", "print", "printing", "green",
    "bull", "bullish", "bullrun", "bull run",
    "pump", "pumping", "pump",
    "run", "running", "rally",
    "diamond", "diamondhand", "💎", "🚀", "spaceship",
    "hodl", "hold the line",
    "stonks", "to the", "lfg",
    "rip faces off", "face-melter", "face melter",
    "buy", "buying", "bought", "call", "calls",
    "hold", "holding", "squeezing", "green", "black", "valhalla",
    "short squeeze", "dd", "improve", "leap", "420", "69",
    "💸", "🙌", "💰", "📈", "🔥", "🤑"
}
bearish = {
    "bagholder", "bagholding",
    "loss", "losses", "lost", "down", "red",
    "bear", "bearish", "crash", "dump", "rug", "rugged",
    "dip", "dipping",
    "paper", "paperhands", "🤲🧻",
    "implode", "collapse", "rekt", "wrecked",
    "bleeding", "bag", "fucked", "destroyed", "screwed",
    "fear", "panic", "selloff",
    "my life is over", "kill me", "i’m done",
    "rip my", "rip me", "rip my calls", "rip my puts",
    "bankrupt", "bk", "chapter 11",
    "margin call", "liquidated", "liquidation",
    "wtf", "omg help", "hell", "pain", "hurts",
    "sell", "selling", "sold", "tank", "tanking",
    "put", "puts", "crashing", "red", "downbad", "crush", "crushing", "crushed",
    "disaster", "exit", "close", "short", "stupid",
    "📉", "💀", "😭", "🥲", "🤡", "🐻", "😵"
}

def custom_sentiment(text):
    t = text.lower()
    score = 0
    score += sum(1 for w in bullish if w in t)
    score -= sum(1 for w in bearish if w in t)
    return score

In [17]:
df['custom_sentiment'] = df['text'].progress_apply(custom_sentiment)
df.head()

100%|██████████| 25862/25862 [00:02<00:00, 9384.93it/s] 


,eastern,text,mentioned_tickers,custom_sentiment
1,2021-01-28 06:32:10-05:00,Math Professor Scott Steiner says the numbers ...,[GME],-2
2,2021-01-28 06:30:35-05:00,Exit the system The CEO of NASDAQ pushed to ha...,"[GME, QQQ]",1
3,2021-01-28 06:28:57-05:00,NEW SEC FILING FOR GME! CAN SOMEONE LESS RETAR...,[GME],0
4,2021-01-28 06:26:56-05:00,"Not to distract from GME, just thought our AMC...","[GME, AMC]",0
6,2021-01-28 06:26:27-05:00,SHORT STOCK DOESN'T HAVE AN EXPIRATION DATE He...,[T],0


In [18]:
df_zero = df[df['custom_sentiment'] == 0]
df_zero.head()

,eastern,text,mentioned_tickers,custom_sentiment
3,2021-01-28 06:28:57-05:00,NEW SEC FILING FOR GME! CAN SOMEONE LESS RETAR...,[GME],0
4,2021-01-28 06:26:56-05:00,"Not to distract from GME, just thought our AMC...","[GME, AMC]",0
6,2021-01-28 06:26:27-05:00,SHORT STOCK DOESN'T HAVE AN EXPIRATION DATE He...,[T],0
10,2021-01-28 06:18:25-05:00,"We need to keep this movement going, we all ca...","[GME, AMC]",0
34,2021-01-28 05:21:46-05:00,They're trying to say this was all done by 'Na...,[RE],0


In [19]:
(df['custom_sentiment'] == 0).mean()

np.float64(0.23756863351635604)

In [20]:
import transformers
import torch
import torch.nn.functional as F

tokenizer = transformers.AutoTokenizer.from_pretrained("ProsusAI/finbert", disable_mmap=True)
model = transformers.AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1924.71it/s, Materializing param=classifier.weight]                                      
BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [23]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def finbert_sentiment(text):
  inputs = tokenizer(text, return_tensors='pt', truncation=True)
  with torch.no_grad():
    logits = model(**inputs).logits
    probs = F.softmax(logits, dim=1).flatten()
    sentiment_score = probs[2] - probs[0]
    return float(sentiment_score)

In [24]:
df['finbert_sentiment'] = df['text'].progress_apply(finbert_sentiment)

100%|██████████| 25862/25862 [32:15<00:00, 13.36it/s]


In [25]:
df.head()

,eastern,text,mentioned_tickers,custom_sentiment,finbert_sentiment
1,2021-01-28 06:32:10-05:00,Math Professor Scott Steiner says the numbers ...,[GME],-2,0.008335
2,2021-01-28 06:30:35-05:00,Exit the system The CEO of NASDAQ pushed to ha...,"[GME, QQQ]",1,0.813214
3,2021-01-28 06:28:57-05:00,NEW SEC FILING FOR GME! CAN SOMEONE LESS RETAR...,[GME],0,0.826607
4,2021-01-28 06:26:56-05:00,"Not to distract from GME, just thought our AMC...","[GME, AMC]",0,0.791619
6,2021-01-28 06:26:27-05:00,SHORT STOCK DOESN'T HAVE AN EXPIRATION DATE He...,[T],0,0.641183


In [28]:
from sklearn.preprocessing import StandardScaler

df['custom_sentiment'] = StandardScaler().fit_transform(df[['custom_sentiment']])
df['finbert_sentiment'] = StandardScaler().fit_transform(df[['finbert_sentiment']])

In [30]:
df = df.drop('custom_sentiment_scaled', axis=1).drop('finbert_sentiment_scaled', axis=1)
df.head()

,eastern,text,mentioned_tickers,custom_sentiment,finbert_sentiment
1,2021-01-28 06:32:10-05:00,Math Professor Scott Steiner says the numbers ...,[GME],-1.320963,-1.622274
2,2021-01-28 06:30:35-05:00,Exit the system The CEO of NASDAQ pushed to ha...,"[GME, QQQ]",-0.032856,0.572957
3,2021-01-28 06:28:57-05:00,NEW SEC FILING FOR GME! CAN SOMEONE LESS RETAR...,[GME],-0.462225,0.609483
4,2021-01-28 06:26:56-05:00,"Not to distract from GME, just thought our AMC...","[GME, AMC]",-0.462225,0.514058
6,2021-01-28 06:26:27-05:00,SHORT STOCK DOESN'T HAVE AN EXPIRATION DATE He...,[T],-0.462225,0.103757


In [ ]:
from Project3.notebooks.parquet_files import REDDIT_WSB_SENTIMENT


df.to_parquet(REDDIT_WSB_SENTIMENT, engine='pyarrow', compression='gzip')